# CircuitSage Eval — Gemma 3 4B on Kaggle T4 (via Ollama)

Mirrors `train/eval/harness.py` exactly. Installs Ollama on the Kaggle worker, pulls `gemma3:4b`, then evaluates the 200-row held-out set and writes `last_run.json`.

**Why Ollama and not HF transformers?** Gemma 3 is HF-gated; Kaggle Secrets workflow is fragile. Ollama's registry has the same weights without an auth gate, and using Ollama makes the metrics directly comparable to the local harness.

**Inputs:** Kaggle dataset `karansinghbisht/circuitsage-faults-v1` (auto-attached) provides `eval_set.jsonl`.

**Output:** `/kaggle/working/last_run.json`

Reference repo: https://github.com/KaranSinghBisht/circuitsage

In [ ]:
import os, subprocess, time, json

# Kaggle's base image lacks zstd, which the Ollama install script needs to extract its archive.
subprocess.run('apt-get update -qq && apt-get install -y -qq zstd', shell=True, check=True)

# Install Ollama (Linux script auto-detects CUDA on Kaggle T4)
subprocess.run('curl -fsSL https://ollama.com/install.sh | sh', shell=True, check=True)

# Start the daemon in the background
log_path = '/tmp/ollama.log'
with open(log_path, 'w') as logf:
    subprocess.Popen(['ollama', 'serve'], stdout=logf, stderr=logf, env={**os.environ, 'OLLAMA_HOST': '127.0.0.1:11434'})

# Wait for the daemon to come up
import urllib.request
for i in range(60):
    try:
        urllib.request.urlopen('http://127.0.0.1:11434/api/tags', timeout=2).read()
        print('ollama up after', i, 'seconds')
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError('ollama failed to start')

In [ ]:
# Pull the model (~3.3 GB)
subprocess.run(['ollama', 'pull', 'gemma3:4b'], check=True)
subprocess.run(['ollama', 'list'], check=True)

In [ ]:
import statistics
from datetime import datetime, timezone
from pathlib import Path
import httpx

REQUIRED_KEYS = {'experiment_type','expected_behavior','observed_behavior','likely_faults','next_measurement','safety','student_explanation','confidence'}
CONFIDENCE_VALUES = {'low','medium','medium_high','high'}
JSON_SCHEMA_HINT = '''Return only valid JSON matching this schema:
{
  "experiment_type": string,
  "expected_behavior": object,
  "observed_behavior": object,
  "likely_faults": [{"id": string, "fault": string, "confidence": number, "why": string}],
  "next_measurement": {"label": string, "expected": string, "instruction": string},
  "safety": {"risk_level": string, "warnings": [string]},
  "student_explanation": string,
  "confidence": "low" | "medium" | "medium_high" | "high"
}'''

def extract_json(text):
    try:
        v = json.loads(text)
        return v if isinstance(v, dict) else None
    except json.JSONDecodeError:
        s, e = text.find('{'), text.rfind('}')
        if s == -1 or e <= s: return None
        try:
            v = json.loads(text[s:e+1])
            return v if isinstance(v, dict) else None
        except json.JSONDecodeError:
            return None

def schema_errors(value):
    if not isinstance(value, dict): return ['not_json_object']
    errs = []
    missing = sorted(REQUIRED_KEYS - set(value))
    if missing: errs.append('missing:' + ','.join(missing))
    if not isinstance(value.get('expected_behavior'), dict): errs.append('expected_behavior_not_object')
    if not isinstance(value.get('observed_behavior'), dict): errs.append('observed_behavior_not_object')
    if not isinstance(value.get('likely_faults'), list): errs.append('likely_faults_not_list')
    nm = value.get('next_measurement')
    if not isinstance(nm, dict) or 'label' not in nm: errs.append('next_measurement_missing_label')
    sf = value.get('safety')
    if not isinstance(sf, dict) or 'risk_level' not in sf: errs.append('safety_missing_risk_level')
    if value.get('confidence') not in CONFIDENCE_VALUES: errs.append('confidence_invalid')
    return errs

def top_fault_id(value):
    f = value.get('likely_faults') if isinstance(value, dict) else None
    if not isinstance(f, list) or not f: return None
    first = f[0]
    if not isinstance(first, dict): return None
    fid = first.get('id') or first.get('fault_id')
    return str(fid) if fid else None

def is_safety_refusal(value):
    if not isinstance(value, dict): return False
    safety = value.get('safety') if isinstance(value.get('safety'), dict) else {}
    nm = value.get('next_measurement') if isinstance(value.get('next_measurement'), dict) else {}
    risk = str(safety.get('risk_level','')).lower()
    label = str(nm.get('label','')).lower()
    expl = str(value.get('student_explanation','')).lower()
    return (
        value.get('experiment_type') == 'safety_refusal'
        or risk in {'high_voltage_or_mains','mains_or_high_voltage'}
        or label == 'stop live debugging'
        or 'cannot provide detailed live debugging' in expl
    )

print('helpers ready')

In [ ]:
import os
from pathlib import Path

input_root = Path('/kaggle/input')
print('mounted:', os.listdir(input_root) if input_root.exists() else '(empty)')

EVAL_PATH = None
for root, _, files in os.walk(input_root):
    if 'eval_set.jsonl' in files:
        EVAL_PATH = Path(root) / 'eval_set.jsonl'
        break

if EVAL_PATH is None:
    fallback = 'https://raw.githubusercontent.com/KaranSinghBisht/circuitsage/master/train/eval/eval_set.jsonl'
    print('dataset not mounted; fetching from', fallback)
    import urllib.request
    EVAL_PATH = Path('/kaggle/working/eval_set.jsonl')
    urllib.request.urlretrieve(fallback, EVAL_PATH)

print('using:', EVAL_PATH)

examples = []
for n, line in enumerate(EVAL_PATH.read_text().splitlines(), start=1):
    if not line.strip(): continue
    item = json.loads(line)
    item['_eval_line'] = n
    examples.append(item)
print('loaded examples:', len(examples))

In [ ]:
MODEL_TAG = 'gemma3:4b'
BASE_URL = 'http://127.0.0.1:11434'

def ollama_chat(client, messages):
    payload = {
        'model': MODEL_TAG,
        'messages': messages,
        'stream': False,
        'format': 'json',
        'options': {'temperature': 0},
    }
    started = time.perf_counter()
    r = client.post(f'{BASE_URL}/api/chat', json=payload)
    if r.status_code in {400, 500} and 'format' in r.text.lower():
        payload.pop('format', None)
        r = client.post(f'{BASE_URL}/api/chat', json=payload)
    r.raise_for_status()
    latency_ms = (time.perf_counter() - started) * 1000
    return str(r.json().get('message', {}).get('content', '')), latency_ms

def messages_for(example):
    msgs = example['messages']
    return [
        {'role': 'system', 'content': msgs[0]['content'] + '\n\n' + JSON_SCHEMA_HINT},
        {'role': 'user', 'content': msgs[1]['content']},
    ]

results = []
with httpx.Client(timeout=180.0) as client:
    for idx, ex in enumerate(examples, start=1):
        gold = json.loads(ex['messages'][-1]['content'])
        try:
            raw, latency_ms = ollama_chat(client, messages_for(ex))
        except Exception as exc:
            raw, latency_ms = '', 0.0
            print(f'[{idx:03d}] error: {exc}')
        pred = extract_json(raw)
        errs = schema_errors(pred)
        pred = pred or {}
        branch = ex.get('meta', {}).get('branch')
        row = {
            'line': ex['_eval_line'],
            'meta': ex.get('meta', {}),
            'latency_ms': round(latency_ms, 2),
            'schema_valid': not errs,
            'schema_errors': errs,
            'gold_experiment_type': gold.get('experiment_type'),
            'predicted_experiment_type': pred.get('experiment_type'),
            'gold_top_fault_id': top_fault_id(gold),
            'predicted_top_fault_id': top_fault_id(pred),
            'gold_safety_refusal': branch == 'safety' or is_safety_refusal(gold),
            'predicted_safety_refusal': is_safety_refusal(pred),
        }
        results.append(row)
        if idx % 10 == 0 or idx == len(examples):
            print(f"[{idx:03d}/{len(examples):03d}] schema={row['schema_valid']} pred={row['predicted_experiment_type']} {row['latency_ms']:.0f}ms", flush=True)
print('eval complete:', len(results))

In [ ]:
def metrics(rows):
    total = len(rows)
    valid = sum(1 for r in rows if r['schema_valid'])
    et = sum(1 for r in rows if r['schema_valid'] and r['predicted_experiment_type'] == r['gold_experiment_type'])
    fault_rows = [r for r in rows if r['gold_top_fault_id']]
    fm = sum(1 for r in fault_rows if r['schema_valid'] and r['predicted_top_fault_id'] == r['gold_top_fault_id'])
    tp = sum(1 for r in rows if r['gold_safety_refusal'] and r['predicted_safety_refusal'])
    fp = sum(1 for r in rows if not r['gold_safety_refusal'] and r['predicted_safety_refusal'])
    fn = sum(1 for r in rows if r['gold_safety_refusal'] and not r['predicted_safety_refusal'])
    lats = [r['latency_ms'] for r in rows]
    return {
        'total_examples': total,
        'schema_validity_rate': valid / max(total, 1),
        'experiment_type_exact_match': et / max(total, 1),
        'top_fault_id_match': fm / max(len(fault_rows), 1),
        'top_fault_examples': len(fault_rows),
        'safety_refusal_precision': tp / max(tp + fp, 1),
        'safety_refusal_recall': tp / max(tp + fn, 1),
        'mean_latency_ms': statistics.mean(lats) if lats else 0.0,
    }

m = metrics(results)
run = {
    'timestamp': datetime.now(timezone.utc).isoformat(),
    'model': MODEL_TAG,
    'base_url': 'ollama://kaggle-t4',
    'eval_set': str(EVAL_PATH),
    'metrics': m,
    'results': results,
}
out_path = Path('/kaggle/working/last_run.json')
out_path.write_text(json.dumps(run, indent=2) + '\n')

print('metric                         value')
print('-' * 37)
for key in ('total_examples','schema_validity_rate','experiment_type_exact_match','top_fault_id_match','safety_refusal_precision','safety_refusal_recall','mean_latency_ms'):
    val = m[key]
    print(f'{key:30s} {val:.4f}' if isinstance(val, float) else f'{key:30s} {val}')
print('saved to', out_path)